In [1]:
import sys
import pandas as pd
import json
import os
import spacy
import matplotlib.pyplot as plt
from jiwer import RemovePunctuation
from dotenv import load_dotenv

!{sys.executable} -m pip install chat_gpt_asr

In [2]:
from chat_gpt_asr.alignment import *

In [3]:
nlp = spacy.load("en_core_web_sm")

In [4]:
load_dotenv()
Root = os.getenv("ROOT_PATH")


In [5]:
sequences = [
    #REF, ASR, LLM
    ("short one here", "shoe order one here", "shorts one her"),
    ("I eat salami pizza salami", "I meat a salami pizza salami", "I eat a big pizza"),
]

In [6]:
ref,asr,llm = sequences[0]
align3(ref, asr, llm)

(['short', '', 'one', 'here'],
 ['shoe', 'order', 'one', 'here'],
 ['shorts', '', 'one', 'her'])

In [7]:
for ref, asr, llm in sequences:
    print_alignment(align3(ref, asr, llm))
    print()

        0      1    2     3
0   short         one  here
1    shoe  order  one  here
2  shorts         one   her

   0     1  2       3      4       5
0  I   eat     salami  pizza  salami
1  I  meat  a  salami  pizza  salami
2  I   eat  a     big  pizza        



In [8]:
#sequences_1 = [
    # REF, ASR, LLM
    #("Ummm I go to school", "Um I go to school", "I go to school"),
#]


In [9]:
path = os.path.join(Root, "results/results-dev-set/results_noisy/results_tiny/results_sentence_confidence_tiny/results_GPT-3.5-Turbo_tiny/gpt-3.5-turbo-0125/results_without_sentence_confidence_tiny/corrected_transcriptions_sentence_confidence_tiny.json")

#path = "/home/mnaderi/Documents/thesis/chat-gpt-asr/results/results-dev-set/results_noisy/results_tiny/results_sentence_confidence_tiny/results_GPT-3.5-Turbo_tiny/gpt-3.5-turbo-0125/results_without_sentence_confidence_tiny/corrected_transcriptions_sentence_confidence_tiny.json"

with open(path, "r") as f:
    data = json.load(f)
    
    transcriptions = [RemovePunctuation()(d["asr_transcription"]["text"]).lower().strip() for d in data]
    reference_transcriptions = [RemovePunctuation()(d["reference_transcription"]).lower().strip() for d in data]
    corrected_transcriptions = [RemovePunctuation()(d["corrected_asr_transcription"]).lower().strip() for d in data]

In [10]:
for i, (ref, asr, llm) in enumerate(zip(reference_transcriptions, transcriptions, corrected_transcriptions)):
    if i > 2000 and i<2010:
        print_alignment(align3(ref, asr, llm))
        print()

    0        1   2       3    4       5   6    7     8         9   ...    37  \
0  one  morning  as   kanti  was  seated  in  his  boat  cleaning  ...  edge   
1  one  morning  as  gandhi  was  seated  in  his  boat  cleaning  ...  edge   
2  one  morning  as  gandhi  was  seated  in  his  boat  cleaning  ...  edge   

     38   39     40    41         42       43  44   45       46  
0  with  two  white        ducklings  clasped  to  her   breast  
1  with  two  white  tuck      links  clasped  to  her  pressed  
2  with  two  white        ducklings  clasped  to  her   breast  

[3 rows x 47 columns]

    0     1    2    3      4     5    6      7    8        9     10         11
0  the  girl  put  the  birds  into  the  water  and  watched  them  anxiously
1  the  girl  put  the  birds  into  the  water  and    watch  them  anxiously
2  the  girl  put  the  birds  into  the  water  and  watched  them  anxiously

        0      1      2      3    4    5   6    7     8         9    10  \

In [11]:
def identify_edit_type(ref, asr, llm, tokenizer_fn=None):
    edit_types = []
    edits = {"Improved":[],"IntroducedError":[],"LeftCorrect":[],"LeftIncorrect":[]}
    rr, aa, ll = align3(ref, asr, llm, tokenizer_fn)
    if len(rr) == len(aa) == len(ll):
        for r,a,l in zip(rr,aa,ll):
            if a == l == r:
                edit_types.append("LeftCorrect")  # left it correct
                edits["LeftCorrect"].append((a,l,r))
            elif a != l and l == r:
                edit_types.append("Improved")  # improve it
                edits["Improved"].append((a,l,r))
            elif a != r and l != r:
                edit_types.append("LeftIncorrect")  # left it incorrect
                edits["LeftIncorrect"].append((a,l,r))
            elif a == r and l != r:
                edit_types.append("IntroducedError")  # introducing an error
                edits["IntroducedError"].append((a,l,r))
    else:
        raise Exception
    return edit_types, edits

In [12]:
tokenizer_fn = lambda s: [token.text for token in nlp(s)]
data_1 = []
data_2 = []
for i, (ref, asr, llm) in enumerate(zip(reference_transcriptions, transcriptions, corrected_transcriptions)):
    #if i > 10:
        #break
    edit_types, edits = identify_edit_type(ref, asr, llm, tokenizer_fn)
    word_counts = {'Improved': 0, 'IntroducedError': 0, 'LeftCorrect': 0, 'LeftIncorrect': 0}
    poses = [token.pos_ for token in nlp(llm)]
    data_2.append(list(zip(edit_types, poses)))
    for edit_type in edit_types:
        word_counts[edit_type] += 1
        
    
    # Create a DataFrame to store word counts by edit type
    data_1.append(word_counts)
    # df_edit_types = pd.DataFrame(word_counts.items(), columns=['Type', 'Count'])
    
    # Display the DataFrame
    # print("asr: {} len:{} \nllm: {} len: {} \nref: {} len: {}".format(asr, len(asr), llm, len(llm), ref, len(ref)))
    # print('edits: ', edits)
    # print(i, df_edit_types , "\n")

In [13]:
pd.DataFrame(data_1)

,Improved,IntroducedError,LeftCorrect,LeftIncorrect
0,1,0,21,0
1,1,0,9,0
2,0,0,13,0
3,2,0,25,0
4,0,0,10,1
...,...,...,...,...
2859,0,0,13,2
2860,0,0,11,0
2861,0,0,20,2
2862,1,0,30,1


In [14]:
import pandas as pd

# Flatten the list of lists
flattened_data = [item for sublist in data_2 for item in sublist]

# Initialize a dictionary to store counts
combination_counts = {}

# Count the occurrences of each unique combination
for operation, pos in flattened_data:
    if (operation, pos) in combination_counts:
        combination_counts[(operation, pos)] += 1
    else:
        combination_counts[(operation, pos)] = 1

# Convert the dictionary into a DataFrame
df = pd.DataFrame(combination_counts.items(), columns=['Combination', 'Count'])

# # Split the 'Combination' column into 'Operation' and 'POS'
df[['Operation', 'POS']] = pd.DataFrame(df['Combination'].tolist(), index=df.index)

# Drop the 'Combination' column
df.drop(columns=['Combination'], inplace=True)

# Pivot the DataFrame to have operations as rows and POS counts as columns
pivot_df = df.pivot_table(index='Operation', columns='POS', values='Count', fill_value=0)

pivot_df.to_csv('final_pos_result.csv')

pivot_df
# # Display the resulting DataFrame
# print(pivot_df)


POS,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,SYM,VERB,X
Operation,,,,,,,,,,,,,,,,,
Improved,116.0,162.0,85.0,49.0,46.0,83.0,2.0,403.0,17.0,36.0,98.0,72.0,0.0,32.0,0.0,284.0,0.0
IntroducedError,28.0,44.0,13.0,18.0,10.0,28.0,0.0,82.0,4.0,7.0,41.0,8.0,0.0,13.0,0.0,74.0,0.0
LeftCorrect,2849.0,4957.0,2601.0,3097.0,1879.0,4221.0,173.0,7285.0,229.0,1295.0,6406.0,648.0,5.0,1297.0,0.0,5884.0,3.0
LeftIncorrect,454.0,570.0,326.0,396.0,272.0,557.0,50.0,1629.0,78.0,125.0,709.0,516.0,0.0,133.0,1.0,1054.0,1.0


In [ ]:
# axis=0 # sum of rows
axis=1 # sum of columns
(pivot_df/pivot_df.sum(axis)*100).round(2)

## debug

In [44]:
data = {
    'Improved': [116.0, 162.0, 85.0, 49.0, 46.0, 83.0, 2.0, 403.0, 17.0, 36.0, 98.0, 72.0, 0.0, 32.0, 0.0, 284.0, 0.0],
    'IntroducedError': [28.0, 44.0, 13.0, 18.0, 10.0, 28.0, 0.0, 82.0, 4.0, 7.0, 41.0, 8.0, 0.0, 13.0, 0.0, 74.0, 0.0],
    'LeftCorrect': [2849.0, 4957.0, 2601.0, 3097.0, 1879.0, 4221.0, 173.0, 7285.0, 229.0, 1295.0, 6406.0, 648.0, 5.0, 1297.0, 0.0, 5884.0, 3.0],
    'LeftIncorrect': [454.0, 570.0, 326.0, 396.0, 272.0, 557.0, 50.0, 1629.0, 78.0, 125.0, 709.0, 516.0, 0.0, 133.0, 1.0, 1054.0, 1.0]
}
df = pd.DataFrame(data, index=['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X'])
df = df.T
df

,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,SYM,VERB,X
Improved,116.0,162.0,85.0,49.0,46.0,83.0,2.0,403.0,17.0,36.0,98.0,72.0,0.0,32.0,0.0,284.0,0.0
IntroducedError,28.0,44.0,13.0,18.0,10.0,28.0,0.0,82.0,4.0,7.0,41.0,8.0,0.0,13.0,0.0,74.0,0.0
LeftCorrect,2849.0,4957.0,2601.0,3097.0,1879.0,4221.0,173.0,7285.0,229.0,1295.0,6406.0,648.0,5.0,1297.0,0.0,5884.0,3.0
LeftIncorrect,454.0,570.0,326.0,396.0,272.0,557.0,50.0,1629.0,78.0,125.0,709.0,516.0,0.0,133.0,1.0,1054.0,1.0


In [51]:
df = df.drop(columns=["X", "SYM", "PUNCT"])

In [62]:
df_column_wise = (df.div(df.sum(axis=1), axis=0)*100).round(2)
df_column_wise
# print(df_column_wise.T.to_latex(float_format="{:.2f}".format))

,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,SCONJ,VERB
Improved,7.81,10.91,5.72,3.30,3.10,5.59,0.13,27.14,1.14,2.42,6.60,4.85,2.15,19.12
IntroducedError,7.57,11.89,3.51,4.86,2.70,7.57,0.00,22.16,1.08,1.89,11.08,2.16,3.51,20.00
LeftCorrect,6.65,11.58,6.07,7.23,4.39,9.86,0.40,17.01,0.53,3.02,14.96,1.51,3.03,13.74
LeftIncorrect,6.61,8.30,4.75,5.77,3.96,8.11,0.73,23.72,1.14,1.82,10.32,7.51,1.94,15.34


In [61]:
df_row_wise = (df.div(df.sum(axis=0), axis=1)*100).round(2)
df_row_wise.T
# print(df_row_wise.T.to_latex(float_format="{:.2f}".format))

,Improved,IntroducedError,LeftCorrect,LeftIncorrect
ADJ,3.37,0.81,82.65,13.17
ADP,2.83,0.77,86.46,9.94
ADV,2.81,0.43,85.98,10.78
AUX,1.38,0.51,86.99,11.12
CCONJ,2.08,0.45,85.14,12.32
DET,1.70,0.57,86.34,11.39
INTJ,0.89,0.00,76.89,22.22
NOUN,4.29,0.87,77.51,17.33
NUM,5.18,1.22,69.82,23.78
PART,2.46,0.48,88.52,8.54


In [17]:
ref = "i don't go to school"
asr = "i not go to school"
ll  = "i don't go to uni all"
align3(ref, asr, ll, tokenizer_fn )

(['i', 'do', "n't", 'go', 'to', '', 'school'],
 ['i', '', 'not', 'go', 'to', '', 'school'],
 ['i', 'do', "n't", 'go', 'to', 'uni', 'all'])

In [1]:
import pandas as pd


In [5]:
df = pd.DataFrame({"A":[1,2,3],"B": [4,5,6]}, index=['a','b','c'])
df

,A,B
a,1,4
b,2,5
c,3,6


In [7]:
sum_of_rows = df.sum(0)
(df/sum_of_rows*100).round(2)

,A,B
a,16.67,26.67
b,33.33,33.33
c,50.00,40.00
